# Employee Emotion Detection — ML Training
Train a Random Forest on TF-IDF + numerical features, save artifacts for Flask API.

In [ ]:
import pandas as pd
import numpy as np
import pickle, os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from scipy.sparse import hstack, csr_matrix

print('Libraries loaded ✓')

## 1. Load Dataset

In [ ]:
# Update path if needed
DATA_PATH = '../backend/Employee_Emotion_Dashboard 1.xlsx'

df = pd.read_excel(DATA_PATH)
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
df.head()

## 2. EDA — Mood Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Mood distribution
mood_counts = df['Mood'].value_counts()
colors = ['#22c55e', '#f59e0b', '#ef4444']
axes[0].bar(mood_counts.index, mood_counts.values, color=colors)
axes[0].set_title('Mood Distribution')
axes[0].set_ylabel('Count')

# Stress level distribution
stress_counts = df['Stress Level'].value_counts()
axes[1].pie(stress_counts.values, labels=stress_counts.index, autopct='%1.1f%%',
            colors=['#22c55e', '#f59e0b', '#ef4444'])
axes[1].set_title('Stress Level Distribution')

# Productivity score by mood
df.boxplot(column='Productivity Score', by='Mood', ax=axes[2],
           boxprops=dict(color='#6366f1'))
axes[2].set_title('Productivity Score by Mood')
axes[2].set_xlabel('Mood')
plt.suptitle('')
plt.tight_layout()
plt.savefig('../backend/static/eda_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print('EDA plots saved ✓')

## 3. Feature Engineering

In [ ]:
STRESS_MAP   = {'Low': 0, 'Medium': 1, 'High': 2}
WORKLOAD_MAP = {'Low': 0, 'Medium': 1, 'High': 2}

# TF-IDF on text statements
vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
tfidf = vectorizer.fit_transform(df['Text Statement'].fillna('').astype(str))

# Numerical features
stress_num     = df['Stress Level'].map(STRESS_MAP).fillna(1).values
workload_num   = df['Workload Level'].map(WORKLOAD_MAP).fillna(1).values
productivity   = df['Productivity Score'].fillna(df['Productivity Score'].median()).values

num_features = np.column_stack([stress_num, workload_num, productivity])
scaler = StandardScaler()
num_scaled = scaler.fit_transform(num_features)

X = hstack([tfidf, csr_matrix(num_scaled)])

# Target
le = LabelEncoder()
y  = le.fit_transform(df['Mood'])

print(f'Feature matrix: {X.shape}')
print(f'Classes: {le.classes_}')

## 4. Train / Evaluate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}\n')
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../backend/static/confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print('Confusion matrix saved ✓')

## 6. Save Model Artifacts

In [ ]:
MODEL_DIR = '../backend/models'
os.makedirs(MODEL_DIR, exist_ok=True)

artifacts = {
    'model':         model,
    'vectorizer':    vectorizer,
    'scaler':        scaler,
    'label_encoder': le,
    'stress_map':    STRESS_MAP,
    'workload_map':  WORKLOAD_MAP,
}

for name, obj in artifacts.items():
    path = os.path.join(MODEL_DIR, f'{name}.pkl')
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'Saved {name}.pkl ✓')

print('\nAll artifacts saved. Ready to run Flask API!')